# Install Dependecies

In [21]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [22]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [23]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import time
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer


# Set `ROOT_DIR`

In [4]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [19]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# 2 Week 36: Data and Rule-Based Baselines
Download the dataset and inspect its columns. Report Item 1 separately by
language and split, plus overall example counts and answerability proportions.
Compute Item 2 from the training questions separately by language. Apply
Item 3 to every answerable example in both splits.

1. report the number of examples, answerable/unanswerable proportions,
median and interquartile range of tokenised question and context lengths
(a table is sufficient; plots are optional), missing values and exact duplicate
question–context pairs;
2. report the five most common question tokens and their counts for each
language, together with an English translation, and explain your tokenisation; and
3. verify programmatically that every answerable item’s answer equals the
substring beginning at answer start, and report the number checked and
any failures.

Implement and evaluate two answerability baselines: (i) the majority-class
baseline estimated from the training split and (ii) a deterministic rule-based
classifier that uses only the question and context. The rule may use tokenisation,
lexical features or machine translation, but no labelled validation examples or
trained answerability/QA model. Discuss what information the rule can and
cannot exploit in this cross-lingual setting.

### 2.1

In [25]:
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [69]:
def token_len(texts):
    return [len(xlm_tokeniser(t)["input_ids"]) for t in texts]

In [70]:
df_train["q_len"] = token_len(df_train["question"])
df_train["c_len"] = token_len(df_train["context"])

In [83]:
df_train_qlen = df_train.sort_values("q_len")
q1 = df_train_qlen["q_len"].quantile(0.25)
q3 = df_train_qlen["q_len"].quantile(0.75)
mask = df_train_qlen["q_len"].between(q1, q3)
iqr_qlen = df_train_qlen.loc[mask, "q_len"]
iqr_qlen

15282    12
15284    12
15208    12
15240    12
15247    12
         ..
15302    17
5340     17
26       17
4        17
5339     17
Name: q_len, Length: 8535, dtype: int64

In [36]:
print(df_train["lang"].unique())

<ArrowStringArray>
['bn', 'ja', 'ko', 'ru', 'fi', 'ar', 'te']
Length: 7, dtype: str


In [77]:
df_train.groupby(["lang"]).count()

,question,context,answerable,answer_start,answer,answer_inlang,q_len,c_len
lang,,,,,,,,
ar,2558,2558,2558,2558,2558,0,2558,2558
bn,2598,2598,2598,2598,2598,50,2598,2598
fi,2126,2126,2126,2126,2126,50,2126,2126
ja,2301,2301,2301,2301,2301,50,2301,2301
ko,2422,2422,2422,2422,2422,0,2422,2422
ru,1983,1983,1983,1983,1983,50,1983,1983
te,1355,1355,1355,1355,1355,50,1355,1355
